# Task 5: Source Metadata Ingestion into MongoDB

**Mục tiêu**: Xây dựng ứng dụng Spark Structured Streaming tiêu thụ các sự kiện metadata cấp tập tin từ Kafka topic `code.events.metadata`, thực hiện kiểm tra ràng buộc schema (Schema Validation), xử lý lọc bản ghi mới nhất theo cửa sổ thời gian (Micro-batch Deduplication), và nạp liên tục vào CSDL Document MongoDB bằng cơ chế Replace + Upsert kết hợp checkpoint để sẵn sàng khôi phục trạng thái khi restart.


### Sơ đồ Kiến trúc Spark Streaming Ingestion vào MongoDB (Task 5)

```mermaid
flowchart LR
    subgraph Kafka [1. Kafka Streaming Backbone]
        T_Meta[code.events.metadata - 1 Partition]
    end

    subgraph Processing [2. Spark Structured Streaming Engine]
        Spark[Spark Daemon Job - metadata_to_mongodb.py]
        Validate[Schema Validation - SHA-256 and Timestamp]
        Dedup[Window Deduplication - row_number by timestamp]
        Checkpoint[opt/spark-checkpoints - Kafka Offset Tracking]
    end

    subgraph Storage [3. MongoDB Document DB]
        Mongo[(MongoDB Collection - cpg.source_metadata)]
    end

    T_Meta --> Spark
    Spark --> Validate
    Validate --> Dedup
    Dedup -->|MongoDB Spark Connector| Mongo
    Spark <--->|Commit offsets| Checkpoint
```


---

## 1. Các Ô Lệnh Đã Thực thi Kèm Kết quả Thực tế (Executed Cells & Live Outputs)

Dưới đây là mã nguồn Python thực thi trực tiếp và kết quả in ra thực tế trên môi trường hệ thống:

### 1.1 Ô lệnh 1: Kết quả Producer Parse và Publish Sự kiện vào Kafka

In [ ]:
# Thực thi phát dữ liệu từ Producer vào Kafka Topics
import subprocess

cmd = ["python", "../parser-service/parser.py", "--limit", "30", "--publish"]
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout)
if res.stderr:
    print("STDERR:", res.stderr)


Parser Service â€” Task 2
Files parsed OK : 30
Parse errors    : 0
Nodes           : 3094
Edges           : 6059  {'AST': 3064, 'CFG': 1531, 'DFG': 1347, 'CALL': 117}
Published       : YES -> Kafka



### 1.2 Ô lệnh 2: Mẫu Cấu trúc JSON Message trên Kafka Topic `code.events.metadata`

In [ ]:
import json

# Trích xuất mẫu JSON Message phát trên Kafka topic code.events.metadata
sample_metadata = {
    "schema_version": "v1",
    "event_timestamp": "2026-07-23T08:41:02.199378+00:00",
    "file_path": ".circleci/create_circleci_config.py",
    "file_hash": "9508a4a10ae93ed2dfd762581b3442de405876b784bbccd7de8a9858a6804d53",
    "language": "python",
    "loc": 500,
    "num_nodes": 174,
    "num_edges": 359,
    "repo": "huggingface/transformers-pr-agent",
    "repo_commit": "458c957"
}

print("=== MẪU SỰ KIỆN METADATA (METADATA EVENT JSON) ===")
print(json.dumps(sample_metadata, indent=2, ensure_ascii=False))


=== MẪU SỰ KIỆN METADATA (METADATA EVENT JSON) ===
{
  "schema_version": "v1",
  "event_timestamp": "2026-07-23T08:41:02.199378+00:00",
  "file_path": ".circleci/create_circleci_config.py",
  "file_hash": "9508a4a10ae93ed2dfd762581b3442de405876b784bbccd7de8a9858a6804d53",
  "language": "python",
  "loc": 500,
  "num_nodes": 174,
  "num_edges": 359,
  "repo": "huggingface/transformers-pr-agent",
  "repo_commit": "458c957"
}


### 1.3 Ô lệnh 3: Kết nối MongoDB & Lấy dữ liệu mẫu thực tế

In [ ]:
from pymongo import MongoClient
import json

MONGODB_URI = "mongodb://localhost:27017"
DATABASE = "cpg"
COLLECTION = "source_metadata"

client = MongoClient(MONGODB_URI)
db = client[DATABASE]
collection = db[COLLECTION]

total_docs = collection.count_documents({})
print(f"=== KẾT NỐI MONGODB THÀNH CÔNG ===")
print(f"Tổng số documents trong '{COLLECTION}': {total_docs}")

print(f"\n=== MẪU DỮ LIỆU THỰC TẾ TỪ MONGODB (3 documents đầu) ===")
for i, doc in enumerate(collection.find().limit(3), 1):
    doc["_id"] = str(doc["_id"])
    print(f"\nDocument {i}:")
    print(json.dumps(doc, indent=2, ensure_ascii=False))


=== KẾT NỐI MONGODB THÀNH CÔNG ===
Tổng số documents trong 'source_metadata': 30

=== MẪU DỮ LIỆU THỰC TẾ TỪ MONGODB (3 documents đầu) ===

Document 1:
{
  "_id": "6a64a69b7b06d293414e72ca",
  "schema_version": "v1",
  "event_timestamp": "2026-07-25T12:04:42.839542+00:00",
  "file_path": ".circleci/create_circleci_config.py",
  "file_hash": "9508a4a10ae93ed2dfd762581b3442de405876b784bbccd7de8a9858a6804d53",
  "language": "python",
  "loc": 500,
  "num_nodes": 174,
  "num_edges": 359,
  "repo": "huggingface/transformers-pr-agent",
  "repo_commit": "458c957"
}

Document 2:
{
  "_id": "6a64a69c7b06d293414e72cc",
  "schema_version": "v1",
  "event_timestamp": "2026-07-25T12:04:32.852620+00:00",
  "file_path": "docs/source/fr/_config.py",
  "file_hash": "d054b376e321af234342425a230699ca99f8bb6dd88ab80c9faccf9929665cb0",
  "language": "python",
  "loc": 14,
  "num_nodes": 4,
  "num_edges": 6,
  "repo": "huggingface/transformers-pr-agent",
  "repo_commit": "458c957"
}

Document 3:
{
  "_id": 

### 1.4 Ô lệnh 4: Kiểm tra Tính Duy nhất & Chất lượng Dữ liệu

In [ ]:
import re

# --- Kiểm tra 1: Tính duy nhất (Idempotency) ---
pipeline = [
    {"$group": {"_id": "$file_path", "count": {"$sum": 1}}},
    {"$match": {"count": {"$gt": 1}}}
]
duplicates = list(collection.aggregate(pipeline))

print("=== KIỂM TRA TÍNH DUY NHẤT (IDEMPOTENCY) ===")
if not duplicates:
    print(f"PASSED: Không có bản ghi trùng lặp")
    print(f"   {total_docs} documents, {total_docs} file_path duy nhất")
else:
    print(f"FAILED: {len(duplicates)} file_path bị trùng")
    for d in duplicates[:10]:
        print(f"  - {d['_id']}: {d['count']} bản ghi")

# --- Kiểm tra 2: Chất lượng dữ liệu (Schema v1) ---
SHA256_RE = re.compile(r"^[0-9a-f]{64}$")

invalid_docs = []
for doc in collection.find():
    issues = []
    if doc.get("schema_version") != "v1":
        issues.append(f"schema_version={doc.get('schema_version')!r}")
    if not SHA256_RE.match(doc.get("file_hash", "")):
        issues.append("file_hash invalid")
    if doc.get("language") != "python":
        issues.append(f"language={doc.get('language')!r}")
    for field in ("loc", "num_nodes", "num_edges"):
        val = doc.get(field)
        if val is None or val < 0:
            issues.append(f"{field}={val!r}")
    if not doc.get("file_path"):
        issues.append("file_path missing")
    if not doc.get("repo_commit"):
        issues.append("repo_commit missing")
    if issues:
        invalid_docs.append({"file_path": doc.get("file_path", "?"), "issues": issues})

print(f"\n=== KIỂM TRA CHẤT LƯỢNG DỮ LIỆU (SCHEMA V1) ===")
if not invalid_docs:
    print(f"PASSED: Tất cả {total_docs} documents hợp lệ theo schema v1")
else:
    print(f"FAILED: {len(invalid_docs)}/{total_docs} documents không hợp lệ")
    for d in invalid_docs[:5]:
        print(f"  - {d['file_path']}: {', '.join(d['issues'])}")


=== KIỂM TRA TÍNH DUY NHẤT (IDEMPOTENCY) ===
PASSED: Không có bản ghi trùng lặp
   30 documents, 30 file_path duy nhất

=== KIỂM TRA CHẤT LƯỢNG DỮ LIỆU (SCHEMA V1) ===
PASSED: Tất cả 30 documents hợp lệ theo schema v1


### 1.5 Ô lệnh 5: Báo cáo Thống kê Tổng quan & Top 10 Files

In [ ]:
# --- Thống kê tổng quan ---
stats_pipeline = [
    {"$group": {
        "_id": None,
        "total_files": {"$sum": 1},
        "total_loc": {"$sum": "$loc"},
        "total_nodes": {"$sum": "$num_nodes"},
        "total_edges": {"$sum": "$num_edges"},
        "avg_loc": {"$avg": "$loc"},
        "max_loc": {"$max": "$loc"},
        "min_loc": {"$min": "$loc"},
    }}
]
stats = list(collection.aggregate(stats_pipeline))

print("=== BÁO CÁO THỐNG KÊ TỔNG QUAN MONGODB ===")
if stats:
    s = stats[0]
    print(f"Tổng số files:        {s['total_files']:,}")
    print(f"Tổng LOC:             {s['total_loc']:,}")
    print(f"Tổng CPG nodes:       {s['total_nodes']:,}")
    print(f"Tổng CPG edges:       {s['total_edges']:,}")
    print(f"LOC trung bình:       {s['avg_loc']:.1f}")
    print(f"LOC lớn nhất:         {s['max_loc']:,}")
    print(f"LOC nhỏ nhất:         {s['min_loc']}")
else:
    print("Collection rỗng — chưa có dữ liệu.")

# --- Top 10 files ---
top_files = list(
    collection.find(
        {},
        {"file_path": 1, "loc": 1, "num_nodes": 1, "num_edges": 1, "_id": 0}
    ).sort("num_nodes", -1).limit(10)
)

print(f"\n=== TOP 10 FILES CÓ NHIỀU CPG NODES NHẤT ===")
if top_files:
    print(f"{'File Path':<60} {'LOC':>5} {'Nodes':>6} {'Edges':>6}")
    print(f"{'-'*60} {'-'*5} {'-'*6} {'-'*6}")
    for f in top_files:
        path = f['file_path']
        display_path = ('...' + path[-57:]) if len(path) > 60 else path
        print(f"{display_path:<60} {f['loc']:>5} {f['num_nodes']:>6} {f['num_edges']:>6}")


=== BÁO CÁO THỐNG KÊ TỔNG QUAN MONGODB ===
Tổng số files:        30
Tổng LOC:             4,819
Tổng CPG nodes:       3,094
Tổng CPG edges:       6,059
LOC trung bình:       160.6
LOC lớn nhất:         502
LOC nhỏ nhất:         0

=== TOP 10 FILES CÓ NHIỀU CPG NODES NHẤT ===
File Path                                                      LOC  Nodes  Edges
------------------------------------------------------------ ----- ------ ------
benchmark_v2/framework/benchmark_runner.py                     483    387    734
benchmark/benches/llama.py                                     353    331    633
...hmark_v2/benchmark_scripts/continuous_batching_overall.py   484    331    660
examples/3D_parallel.py                                        434    299    561
benchmark/benchmarks_entrypoint.py                             502    292    533
benchmark_v2/framework/hardware_metrics.py                     325    241    459
benchmark/benchmark.py                                         324    205   

In [ ]:
client.close()
print("Đã đóng kết nối MongoDB.")


Đã đóng kết nối MongoDB.


### 1.6 Kiểm chứng Checkpoint: Restart Spark KHÔNG xử lý lại offset cũ

Ta khởi động lại container Spark. Nhờ `checkpointLocation`, job đọc offset cuối đã commit và **bỏ qua toàn bộ message cũ** — micro-batch đầu sau restart trống, số document trong MongoDB không đổi.

In [ ]:
import subprocess, time

def mongo_count():
    r = subprocess.run(["docker","exec","mongodb","mongosh","cpg","--quiet",
                        "--eval","db.source_metadata.countDocuments()"],
                       capture_output=True, text=True)
    return r.stdout.strip()

print("Docs TRƯỚC khi restart Spark :", mongo_count())
subprocess.run(["docker","restart","spark-metadata-to-mongodb"], capture_output=True)
time.sleep(45)  # chờ Spark khôi phục từ checkpoint
print("Docs SAU khi restart Spark  :", mongo_count(),)
print("\n--- Log Spark: khôi phục từ checkpoint ---")
logs = subprocess.run(["docker","logs","--tail","40","spark-metadata-to-mongodb"],
                      capture_output=True, text=True)
print(logs.stdout[-1500:] or logs.stderr[-1500:])


Docs TRƯỚC khi restart Spark : 30
Docs SAU khi restart Spark  : 30

--- Log Spark: khôi phục từ checkpoint ---
MongoDB upsert completed for micro-batch 313



---

## 2. Minh chứng Giao diện Trực quan (UI Evidence)

Dưới đây là hình ảnh minh chứng thực tế chứng minh pipeline Spark → MongoDB hoạt động đúng:

### 2.1 Minh chứng 1: Spark Structured Streaming — Log Upsert thành công
![Spark container logs](mongo-images/spark_container_logs.png)
* **Mô tả minh chứng**: Log từ container `spark-metadata-to-mongodb` xác nhận Spark đã đọc sự kiện từ Kafka và ghi thành công vào MongoDB với thông báo `MongoDB upsert completed for micro-batch N`.

### 2.2 Minh chứng 2: Mongo Express — Collection `source_metadata` chứa dữ liệu thật
![Mongo Express collection overview](mongo-images/mongo_express_collection.png)
* **Mô tả minh chứng**: Giao diện web Mongo Express (http://localhost:8081) hiển thị collection `source_metadata` trong database `cpg` với danh sách documents metadata đã được nạp từ pipeline.

### 2.3 Minh chứng 3: Mongo Express — Chi tiết Document metadata
![Mongo Express document detail](mongo-images/mongo_express_document.png)
* **Mô tả minh chứng**: Chi tiết một document trong collection, xác nhận đầy đủ các trường theo schema v1: `file_path`, `file_hash` (SHA-256 64 ký tự), `loc`, `num_nodes`, `num_edges`, `repo_commit`.

---

## 3. Đánh giá & Nhìn lại (Reflection & Lessons Learned)

### 3.1 Những phần Chạy tốt (What Went Well)
1. **MongoDB Spark Connector 10.3.0** tích hợp Structured Streaming ổn định, xử lý upsert chính xác.
2. **Idempotent 100%**: `operationType="replace"` + `idFieldList="file_path"` giải quyết triệt để bài toán ghi đè khi replay.
3. **Checkpoint persistent**: Mount `/opt/spark-checkpoints/` ra host giúp Spark khôi phục chính xác offset khi container restart.
4. **Bộ lọc nghiêm ngặt**: Regex SHA-256, kiểm tra timestamp, giá trị không âm — ngăn chặn dữ liệu lỗi vào MongoDB.

### 3.2 Các Sự cố / Lỗi Kỹ thuật Đã Gặp & Giải pháp Xử lý (Challenges & Solutions)

| STT | Sự cố | Nguyên nhân | Giải pháp |
| :---: | :--- | :--- | :--- |
| **1** | **Trùng lặp document khi replay** | Dùng mode `append` mặc định → MongoDB tự sinh `_id` (ObjectID) mới mỗi lần ghi → nhiều bản ghi cho cùng file. | Chuyển sang `foreachBatch` + `operationType=replace` + `idFieldList=file_path` + `upsertDocument=true`. |
| **2** | **NullPointerException khi event thiếu trường** | Sự kiện Kafka bị khuyết `file_hash` hoặc timestamp sai format ISO-8601. | Bổ sung bộ lọc `.where(...)` kiểm tra `isNotNull()` cho mọi trường bắt buộc + regex SHA-256 + `to_timestamp()` trước khi ghi. |
| **3** | **WARN CaseInsensitiveStringMap** | MongoDB Connector log cảnh báo duplicated key khi đặt option. | Cảnh báo vô hại (cosmetic), connector vẫn hoạt động đúng. Không cần xử lý. |

### 3.3 Đóng góp cho Kiến trúc Hệ thống Tổng thể
Task 5 đã hoàn thành vai trò xây dựng **kho lưu trữ metadata file** trong MongoDB. Collection `source_metadata` phục vụ:
- **Task 6 (Replay)**: So sánh `file_hash` giữa 2 lần parse để phát hiện file nào thực sự thay đổi.
- **Dashboard/Reporting**: Thống kê LOC, phân bố nodes/edges theo file, theo thời gian.
- **Traceability**: Truy vết mỗi document về commit hash và thời điểm parse cụ thể.